In [1]:
# ============================================================
# WEEK 2: Contextual Data Fusion & Feature Engineering
# Project: Contextual Predictive Maintenance (IoT Edge AI)
# Intern : Sanjay R | Infotact DS/ML Internship | 2026
# ============================================================

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

print("All libraries loaded!")

All libraries loaded!


In [3]:
df = pd.read_csv(r'D:\predictive_maintenance\data\processed\processed_data.csv')
print("Week 1 data loaded!")
print("Shape:", df.shape)
df.head()

Week 1 data loaded!
Shape: (9991, 72)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,process_lag2,rotational_lag1,rotational_lag2,torque_lag1,torque_lag2,tool_lag1,tool_lag2,ambient_humidity,factory_load_index,shift_period
0,10,M14869,M,298.5,309.0,1741,28.0,21,0,0,...,308.6,1667.0,1527.0,28.6,40.2,18.0,16.0,52.472407,32.513293,1
1,11,H29424,H,298.4,308.9,1782,23.9,24,0,0,...,308.7,1741.0,1667.0,28.0,28.6,21.0,18.0,87.042858,47.111035,1
2,12,H29425,H,298.6,309.1,1423,44.3,29,0,0,...,309.0,1782.0,1741.0,23.9,28.0,24.0,21.0,73.919637,81.873799,1
3,13,M14872,M,298.6,309.1,1339,51.1,34,0,0,...,308.9,1423.0,1782.0,44.3,23.9,29.0,24.0,65.919509,12.461421,1
4,14,M14873,M,298.6,309.2,1742,30.0,37,0,0,...,309.1,1339.0,1423.0,51.1,44.3,34.0,29.0,39.361118,85.765599,1


In [4]:
import datetime

base_time = datetime.datetime(2024, 1, 1, 6, 0, 0)
df['timestamp'] = [base_time + datetime.timedelta(minutes=i)
                   for i in range(len(df))]

df['hour']      = df['timestamp'].dt.hour
df['day']       = df['timestamp'].dt.day
df['dayofweek'] = df['timestamp'].dt.dayofweek

print("Timestamps added!")
print(df[['timestamp','hour','day','dayofweek']].head())

Timestamps added!
            timestamp  hour  day  dayofweek
0 2024-01-01 06:00:00     6    1          0
1 2024-01-01 06:01:00     6    1          0
2 2024-01-01 06:02:00     6    1          0
3 2024-01-01 06:03:00     6    1          0
4 2024-01-01 06:04:00     6    1          0


In [5]:
np.random.seed(42)
n = len(df)

# Weather signals
df['wind_speed_kmh']       = np.random.uniform(0, 40, n).round(2)
df['outdoor_temp_celsius'] = np.random.uniform(20, 45, n).round(2)
df['power_grid_voltage']   = np.random.uniform(210, 240, n).round(2)

# Factory signals
df['operator_experience']  = np.random.randint(1, 10, n)
df['is_weekend']           = df['dayofweek'].apply(
                               lambda x: 1 if x >= 5 else 0)

print("External signals added!")
print("Shape:", df.shape)

External signals added!
Shape: (9991, 81)


In [6]:
# Combine sensors to create smarter features

# 1. Temperature difference
df['temp_difference']  = (df['Process temperature [K]'] -
                          df['Air temperature [K]'])

# 2. Power consumption
df['power_consumption'] = (df['Rotational speed [rpm]'] *
                           df['Torque [Nm]'] / 9550)

# 3. Tool wear rate
df['wear_per_rotation'] = (df['Tool wear [min]'] /
                           (df['Rotational speed [rpm]'] + 1))

# 4. Heat stress index
df['heat_stress_index'] = (df['outdoor_temp_celsius'] *
                           df['factory_load_index'] / 100)

# 5. Fatigue risk
df['fatigue_risk']      = ((1 - df['shift_period']) *
                           (10 - df['operator_experience']))

print("Smart features created!")
print("Shape:", df.shape)
print("\nNew features added:")
print("  temp_difference   → process temp minus air temp")
print("  power_consumption → speed x torque / 9550")
print("  wear_per_rotation → tool wear per rotation")
print("  heat_stress_index → outdoor temp x factory load")
print("  fatigue_risk      → night shift x low experience")

Smart features created!
Shape: (9991, 86)

New features added:
  temp_difference   → process temp minus air temp
  power_consumption → speed x torque / 9550
  wear_per_rotation → tool wear per rotation
  heat_stress_index → outdoor temp x factory load
  fatigue_risk      → night shift x low experience
